# Notebook 02: The Agent Loop

**Learning objectives:**
- Understand what makes a system 'agentic' vs. a standard ML pipeline
- See how tool calling works at the API level (no framework)
- Read the agent's reasoning trace and understand each decision
- Observe how the agent adapts to a different scene

**Prerequisites:** Run Notebook 00 and 01 first (or at least 00).

In [ ]:
import sys, os
from pathlib import Path
# Move to project root so relative data paths resolve correctly
_here = Path(os.getcwd())
if not (_here / 'data' / 'scenes').exists() and (_here.parent / 'data' / 'scenes').exists():
    os.chdir(_here.parent)
sys.path.insert(0, str(Path(os.getcwd())))

import inspect
from agent.scene import load_scene
from agent.types import BoundingBox
from agent import tools
from agent.schemas import TOOL_SCHEMAS
from agent.loop import run_agent, SYSTEM_PROMPT
from agent.mock import run_mock_agent

load_scene('scene_a')
print('Scene A loaded. Ready.')

## Step 1: One Tool Example

Before the agent, let's confirm: tools are just functions.

In [ ]:
bb = BoundingBox(0, 200, 0, 200)
result = tools.compute_ndvi(bb)
print(result.summary)

## Step 2: One Schema Example

This is how the model learns what `compute_ndvi` accepts:

In [ ]:
import json
ndvi_schema = next(s for s in TOOL_SCHEMAS if s['name'] == 'compute_ndvi')
print(json.dumps(ndvi_schema, indent=2))

## Step 3: The System Prompt

Read this carefully — every instruction shapes agent behavior.

Note especially:
- *"Explain your reasoning before each tool call"* → creates the trace we learn from
- *"Only call tools when warranted"* → prevents exhaustive tool use
- *"Express uncertainty when appropriate"* → prevents overconfident reports

In [ ]:
print(SYSTEM_PROMPT)

## Step 4: The Loop

The entire agent loop is ~50 lines. Read it:

In [ ]:
from agent.loop import run_agent
print(inspect.getsource(run_agent))

## Step 5: Scripted Run — Scene A

We use **mock mode** here so the trace is identical every time. The model responses are pre-recorded, but **all tool calls are real** (live numpy computation).

Watch each `[TOOL CALL]` block:
1. What did the agent observe that prompted this call?
2. What new information did it learn?
3. What might it do next?

In [ ]:
load_scene('scene_a')
run_mock_agent('scene_a')

### Discussion Questions

1. After `compute_ndvi`: Why did the agent choose `flag_anomalous_regions` next rather than immediately calling `compute_ndwi`?
2. After `flag_anomalous_regions`: The agent now calls `compute_ndwi` on the *anomalous sub-region*, not the full scene. Why is this more efficient?
3. After `compute_ndwi`: The NDWI is negative. Could this mean water stress, bare soil, or something else? What rules this out?
4. After `get_pixel_timeseries`: Why does recent onset matter for diagnosis?
5. The agent called `compare_to_baseline` last, not first. Was this the right order?

## Step 6: Live Run — Scene B

Scene B is ambiguous: mild uniform NDVI depression, no strong spatial anomaly, no baseline.

**Watch:** Does the agent take a different path than it did for Scene A?

Warning: **Requires `ANTHROPIC_API_KEY`** in your environment. Estimated cost: ~$0.10–0.20 for this run.

If you don't have an API key, skip this cell — the take-home exercises cover this.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # load our API key into our environment

# LIVE API CALL
if os.environ.get('ANTHROPIC_API_KEY'):
    load_scene('scene_b')
    run_agent(
        'Assess crop health for the entire scene. '
        'Identify any areas of concern and determine likely causes.',
        temperature=0,
    )
else:
    print('ANTHROPIC_API_KEY not set — skipping live run.')
    print('Set your key and re-run this cell to see the live agent in action.')